## Pytorch

In [28]:
import numpy as np
import pandas as pd
import librosa
import torch
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
import torch.nn as nn
from pathlib import Path

In [2]:
base_path = Path("..")

processed_path = base_path / "data" / "processed"
wet_path = processed_path / "wet"
data_split_path = processed_path / "data_split"

train_path = data_split_path / "train.csv"
val_path = data_split_path / "val.csv"
test_path = data_split_path / "test.csv"

In [3]:
train_df = pd.read_csv(train_path)

train_df.head()

,source_id,wet_path,room_size,wet_level,rate_hz,depth,room_size_norm,wet_level_norm,rate_hz_norm,depth_norm
0,Bridge_1-0,..\data\processed\wet\Bridge_1-0_v0.wav,0.374540,0.570429,3.793973,0.598658,0.374540,0.950714,0.731994,0.598658
1,Bridge_1-0,..\data\processed\wet\Bridge_1-0_v1.wav,0.156019,0.093597,0.761376,0.866176,0.156019,0.155995,0.058084,0.866176
2,Bridge_1-0,..\data\processed\wet\Bridge_1-0_v2.wav,0.601115,0.424844,0.592630,0.969910,0.601115,0.708073,0.020584,0.969910
3,Bridge_1-0,..\data\processed\wet\Bridge_1-0_v3.wav,0.832443,0.127403,1.318212,0.183405,0.832443,0.212339,0.181825,0.183405
4,Bridge_1-0,..\data\processed\wet\Bridge_1-0_v4.wav,0.304242,0.314854,2.443753,0.291229,0.304242,0.524756,0.431945,0.291229


In [4]:
audio_path = train_df.loc[0, "wet_path"]

y, sr = librosa.load(
    audio_path,
    sr=None
)

print(audio_path)
print(y.shape)
print(sr)

c:\Users\User\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


..\data\processed\wet\Bridge_1-0_v0.wav
(240000,)
48000


#### Матрица частот и временных отрезков

In [5]:
mel_spec = librosa.feature.melspectrogram(
    y=y,
    sr=sr,
    n_fft=2048,
    hop_length=512,
    n_mels=128
)

mel_spec.shape

(128, 469)

In [6]:
mel_db = librosa.power_to_db(
    mel_spec,
    ref=np.max
)

print(mel_db.shape)
print(mel_db.min())
print(mel_db.max())

(128, 469)
-80.0
0.0


#### Нормирование

In [7]:
mel_norm = (mel_db + 80) / 80
print(mel_norm.min())
print(mel_norm.max())
print(mel_norm.shape)

0.0
1.0
(128, 469)


#### Тензоры

In [8]:
mel_tensor = torch.from_numpy(mel_norm).unsqueeze(0)

print(mel_tensor.shape)
mel_tensor.dtype

torch.Size([1, 128, 469])


torch.float32

#### Класс для перевода в тензоры

In [24]:
class GuitarFXDataset(Dataset):

    def __init__(self, df):
        self.df = df

    def __len__(self):
        return len(self.df)

    def __getitem__(self, index: int):
        row = self.df.iloc[index]
        audio_path = row["wet_path"]

        y, sr = librosa.load(
            audio_path,
            sr=None
        )


        target_length = 5 * sr

        if len(y) > target_length:
            y = y[:target_length]

        elif len(y) < target_length:
            y = np.pad(y, (0, target_length - len(y)))


        mel_spec = librosa.feature.melspectrogram(
            y=y,
            sr=sr,
            n_fft=2048,
            hop_length=512,
            n_mels=128
        )

        mel_db = librosa.power_to_db(
            mel_spec,
            ref=np.max
        )

        mel_norm = (mel_db + 80) / 80

        mel_tensor = torch.from_numpy(mel_norm).unsqueeze(0)

        targets_cols = [
            "room_size_norm",
            "wet_level_norm",
            "rate_hz_norm",
            "depth_norm"
        ]   

        target = row[targets_cols].to_numpy(dtype=np.float32)
        target_tensor = torch.from_numpy(target)


        return mel_tensor, target_tensor

In [25]:
train_dataset = GuitarFXDataset(train_df)

x, y = train_dataset[0]

print(x.shape)
print(x.dtype)

print(y.shape)
print(y.dtype)
print(y)




torch.Size([1, 128, 469])
torch.float32
torch.Size([4])
torch.float32
tensor([0.3745, 0.9507, 0.7320, 0.5987])


#### Data loader

In [26]:

train_loader = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True
)

x_batch, y_batch = next(iter(train_loader))

print(x_batch.shape)
print(y_batch.shape)

torch.Size([16, 1, 128, 469])
torch.Size([16, 4])


In [30]:
conv1 = nn.Conv2d(
    in_channels=1,
    out_channels=16,
    kernel_size=3,
    padding=1
)

out = conv1(x_batch)

print(x_batch.shape)
print(out.shape)

torch.Size([16, 1, 128, 469])
torch.Size([16, 16, 128, 469])


#### Выделяю самые важные признаки

In [ ]:
pool = nn.MaxPool2d(kernel_size=2)
pooled = pool(out)

print(out.shape)
print(pooled.shape)

torch.Size([16, 16, 128, 469])
torch.Size([16, 16, 64, 234])


#### Нелинейность (-n -> 0)

In [33]:
relu = nn.ReLU()
activated = relu(out)

print(out.shape)
print(activated.shape)

torch.Size([16, 16, 128, 469])
torch.Size([16, 16, 128, 469])


#### Все вместе

In [35]:
conv_out = conv1(x_batch)
relu_out = relu(conv_out)
pool_out = pool(relu_out)

x_batch.shape, conv_out.shape, relu_out.shape, pool_out.shape

(torch.Size([16, 1, 128, 469]),
 torch.Size([16, 16, 128, 469]),
 torch.Size([16, 16, 128, 469]),
 torch.Size([16, 16, 64, 234]))

In [37]:
conv2 = nn.Conv2d(
    in_channels=16,
    out_channels=32,
    kernel_size=3,
    padding=1
)

relu2 = nn.ReLU()
pool2 = nn.MaxPool2d(kernel_size=2)

conv2_out = conv2(pool_out)
relu2_out = relu2(conv2_out)
pool2_out = pool2(relu2_out)

print(conv2_out.shape)
print(relu2_out.shape)
print(pool2_out.shape)


torch.Size([16, 32, 64, 234])
torch.Size([16, 32, 64, 234])
torch.Size([16, 32, 32, 117])


#### Развертывание

In [38]:
flatten = nn.Flatten()
flat_out = flatten(pool2_out)

print(pool2_out.shape)
print(flat_out.shape)

torch.Size([16, 32, 32, 117])
torch.Size([16, 119808])


In [40]:
conv3 = nn.Conv2d(
    in_channels=32,
    out_channels=64,
    kernel_size=3,
    padding=1
)

relu3 = nn.ReLU()
pool3 = nn.MaxPool2d(kernel_size=2)

conv3_out = conv3(pool2_out)
relu3_out = relu3(conv3_out)
pool3_out = pool3(relu3_out)

print(pool3_out.shape)

torch.Size([16, 64, 16, 58])


In [41]:
flat3 = flatten(pool3_out)

print(pool3_out.shape)
print(flat3.shape)

torch.Size([16, 64, 16, 58])
torch.Size([16, 59392])


In [42]:
adaptive_pool = nn.AdaptiveAvgPool2d((4, 4))

adaptive_out = adaptive_pool(pool3_out)

print(pool3_out.shape)
print(adaptive_out.shape)

torch.Size([16, 64, 16, 58])
torch.Size([16, 64, 4, 4])


In [46]:
flat = nn.Flatten()
flat_out = flat(adaptive_out)

fc1 = nn.Linear(
    in_features=1024,
    out_features=128
)

fc_relu = nn.ReLU()

hidden = fc_relu(fc1(flat_out))

fc2 = nn.Linear(
    in_features = 128,
    out_features=4
)

preds = fc2(hidden)

print(flat_out.shape)
print(hidden.shape)
print(preds.shape)
print(preds[0])

torch.Size([16, 1024])
torch.Size([16, 128])
torch.Size([16, 4])
tensor([-0.1189, -0.0336,  0.0384,  0.0427], grad_fn=<SelectBackward0>)


#### Модель

In [69]:
class GuitarFXCNN(nn.Module):

    def __init__(self):
        super().__init__()

        self.conv1 = nn.Conv2d(
            in_channels=1,
            out_channels=16,
            kernel_size=3,
            padding=1
        )

        self.relu1 = nn.ReLU()
        self.pool1 = nn.MaxPool2d(kernel_size=2)

        self.conv2 = nn.Conv2d(
            in_channels=16,
            out_channels=32,
            kernel_size=3,
            padding=1
        )

        self.relu2 = nn.ReLU()
        self.pool2 = nn.MaxPool2d(kernel_size=2)

        self.conv3 = nn.Conv2d(
            in_channels=32,
            out_channels=64,
            kernel_size=3,
            padding=1
        )

        self.relu3 = nn.ReLU()
        self.pool3 = nn.MaxPool2d(kernel_size=2)

        self.adaptive_pool = nn.AdaptiveAvgPool2d((4, 4))

        self.flat = nn.Flatten()

        self.fc1 = nn.Linear(
            in_features=1024,
            out_features=128
        )

        self.fc_relu = nn.ReLU()

        self.fc2 = nn.Linear(
            in_features=128,
            out_features=4
        )

        


    def forward(self, x):

        x = self.conv1(x)
        x = self.relu1(x)
        x = self.pool1(x)

        x = self.conv2(x)
        x = self.relu2(x)
        x = self.pool2(x)

        x = self.conv3(x)
        x = self.relu3(x)
        x = self.pool3(x)

        x = self.adaptive_pool(x)
        x = self.flat(x)

        x = self.fc1(x)
        x = self.fc_relu(x)

        x = self.fc2(x)

        return x

In [70]:
model = GuitarFXCNN()
out = model(x_batch)

print(x_batch.shape)
print(out.shape)

torch.Size([16, 1, 128, 469])
torch.Size([16, 4])
